In [4]:
import glob
import math
import pickle
import types
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)
from torch.utils.data import DataLoader, Dataset

torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
pl.seed_everything(SEED)

TRAIN_DIR = r"C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\train"
VAL_DIR = r"C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\valid"

CKPT = {
    "SimCLR+head": r"C:\Users\Administrator\Desktop\Coding\SimCLR_VIT_clf.pth",
    "CE": r"C:\Users\Administrator\Desktop\Coding\CE_loss_VIT_clf.pth",
    "ArcFace": r"C:\Users\Administrator\Desktop\Coding\ArcFace_loss_VIT_clf.pth",
    "CAC": r"C:\Users\Administrator\Desktop\Coding\CAC_loss_VIT_clf.pth",
}

IMG_SIZE = 224          # modellar o'qitilgan o'lcham bilan bir xil bo'lsin
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # o'qitishdagi bilan bir xil bo'lsin
BATCH = 64
NUM_WORKERS = 0         # Windows + notebookda 0 xavfsiz
KNN_K, KNN_T = 20, 0.07
N_BOOT = 200            # bootstrap sinov soni (F1 uchun ishonch oralig'i)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
OUT = Path("report")
OUT.mkdir(exist_ok=True)


def make_vit(num_classes=0, pretrained=False):
    """CE va SimCLR+head modellari arxitekturasi. O'qitishdagi bilan bir xil bo'lishi kerak."""
    return timm.create_model("vit_tiny_patch16_224", pretrained=pretrained, num_classes=num_classes)


# %% 2. Dataset (o'zingizdagi ClfDS bo'lsa, shu cell'ni o'chirib, o'zingiznikini ishlating;
#                 u class_to_idx va transform parametrlarini qabul qilishi kerak)
class ClfDS(Dataset):
    def __init__(self, root, class_to_idx=None, transform=None):
        root = Path(root)
        classes = sorted(p.name for p in root.iterdir() if p.is_dir())
        self.class_to_idx = class_to_idx or {c: i for i, c in enumerate(classes)}
        self.id2label = {i: c for c, i in self.class_to_idx.items()}
        missing = [c for c in classes if c not in self.class_to_idx]
        if missing:
            print(f"[ogohlantirish] {root.name}: mappingda yo'q classlar o'tkazib yuborildi: {missing[:5]}")
        self.samples = [
            (p, self.class_to_idx[c])
            for c in classes if c in self.class_to_idx
            for p in sorted((root / c).rglob("*")) if p.suffix.lower() in IMG_EXT
        ]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, y = self.samples[i]
        img = Image.open(p).convert("RGB")
        return (self.transform(img) if self.transform else img), y


val_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = ClfDS(TRAIN_DIR, transform=val_tf)                         # galereya (augmentatsiyasiz)
val_ds = ClfDS(VAL_DIR, class_to_idx=train_ds.class_to_idx, transform=val_tf)
assert len(train_ds) > 0 and len(val_ds) > 0, "Dataset bo'sh: yo'llarni tekshiring"

N_CLS = len(train_ds.class_to_idx)
CLASSES = [train_ds.id2label[i] for i in range(N_CLS)]
print(f"train: {len(train_ds)} rasm | val: {len(val_ds)} rasm | classlar: {N_CLS}")

gal_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)


# %% 3. ArcFace va CAC (o'qitishdagi bilan AYNAN bir xil bo'lishi shart, aks holda checkpoint yuklanmaydi)
class ArcFaceLoss(nn.Module):
    def __init__(self, emb_dim, n_cls, s=30.0, m=0.5):
        super().__init__()
        self.n_cls, self.s, self.m = n_cls, s, m
        self.W = nn.Parameter(torch.empty(n_cls, emb_dim))
        nn.init.xavier_uniform_(self.W)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def cosine(self, emb):
        return F.linear(F.normalize(emb.float()), F.normalize(self.W.float()))

    def forward(self, emb, y):
        cos = self.cosine(emb).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt(1 - cos ** 2)
        phi = cos * math.cos(self.m) - sin * math.sin(self.m)
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        onehot = F.one_hot(y, self.n_cls).bool()
        return F.cross_entropy(self.s * torch.where(onehot, phi, cos), y)


class CACLoss(nn.Module):
    def __init__(self, n_cls, alpha=10.0, lam=0.1):
        super().__init__()
        self.n_cls, self.lam = n_cls, lam
        self.register_buffer("anchors", torch.eye(n_cls) * alpha)

    def dist(self, emb):
        return torch.cdist(emb.float(), self.anchors)

    def forward(self, emb, y):
        d = self.dist(emb)
        onehot = F.one_hot(y, self.n_cls).bool()
        d_true = d.gather(1, y[:, None])
        tuplet = torch.log1p(torch.exp(d_true - d).masked_fill(onehot, 0).sum(1)).mean()
        return tuplet + self.lam * d_true.mean()


class LitArcFace(pl.LightningModule):
    def __init__(self, n_cls, lr=3e-4, emb_dim=128, pretrained=True):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = timm.create_model("vit_tiny_patch16_224", pretrained=pretrained, num_classes=0)
        self.proj = nn.Sequential(nn.Linear(192, emb_dim), nn.BatchNorm1d(emb_dim))
        self.crit = ArcFaceLoss(emb_dim, n_cls)

    def forward(self, x):
        return self.proj(self.backbone(x))


class LitCAC(pl.LightningModule):
    def __init__(self, n_cls, lr=3e-4, pretrained=True):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = timm.create_model("vit_tiny_patch16_224", pretrained=pretrained, num_classes=0)
        self.proj = nn.Linear(192, n_cls)
        self.crit = CACLoss(n_cls)

    def forward(self, x):
        return self.proj(self.backbone(x))


# %% 4. Checkpoint o'qish: klassi yo'q (masalan ViTClf) pickle'larni ham ochadi
class _StubUnpickler(pickle.Unpickler):
    """Topilmagan klass o'rniga bo'sh nn.Module qo'yadi; vaznlar baribir saqlanadi."""

    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except (AttributeError, ImportError):
            print(f"  [stub] {module}.{name} topilmadi, bo'sh klass bilan almashtirildi")
            return type(name, (nn.Module,), {})


_stub_pickle = types.ModuleType("stub_pickle")
_stub_pickle.Unpickler = _StubUnpickler
_stub_pickle.load = pickle.load
_stub_pickle.loads = pickle.loads
_stub_pickle.dump = pickle.dump
_stub_pickle.dumps = pickle.dumps
_stub_pickle.HIGHEST_PROTOCOL = pickle.HIGHEST_PROTOCOL


def read_state_dict(path):
    ck = torch.load(path, map_location="cpu", weights_only=False, pickle_module=_stub_pickle)
    if isinstance(ck, nn.Module):
        print(f"  pickle qilingan modul: {type(ck).__name__}")
        sd = ck.state_dict()
    elif isinstance(ck, dict):
        sd = ck
        for key in ("state_dict", "model_state_dict", "model", "net"):
            if key in ck:
                sd = ck[key].state_dict() if isinstance(ck[key], nn.Module) else ck[key]
                break
    else:
        raise RuntimeError(f"{path}: noma'lum checkpoint turi: {type(ck)}")
    return {k: v for k, v in sd.items() if torch.is_tensor(v)}


def timm_classifier_from_sd(sd, name):
    """Umumiy state_dict dan timm ViT-tiny + Linear head quradi (prefikslarni o'zi topadi)."""
    # head: oxirgi 2D 'weight', shape[0] == class soni
    w_keys = [k for k, v in sd.items() if k.endswith("weight") and v.ndim == 2 and v.shape[0] == N_CLS]
    if not w_keys:
        last2d = [(k, tuple(v.shape)) for k, v in sd.items() if v.ndim == 2][-3:]
        raise RuntimeError(
            f"[{name}] head topilmadi (N_CLS={N_CLS}). Oxirgi 2D qatlamlar: {last2d}. "
            "Model boshqa class soni bilan o'qitilgan bo'lishi mumkin.")
    wk = w_keys[-1]
    bk = wk[:-len("weight")] + "bias"
    anchor = next((k for k in sd if k.endswith("patch_embed.proj.weight")), None)
    if anchor is None:
        raise RuntimeError(f"[{name}] timm ViT kalitlari topilmadi. Birinchi kalitlar: {list(sd)[:6]}. "
                           "Arxitektura timm vit_tiny emas, make_vit() ni moslang.")
    pre = anchor[:-len("patch_embed.proj.weight")]
    new = {k[len(pre):]: v for k, v in sd.items() if k.startswith(pre) and k not in (wk, bk)}
    new["head.weight"] = sd[wk]
    if bk in sd:
        new["head.bias"] = sd[bk]
    dropped = [k for k in sd if not k.startswith(pre) and k not in (wk, bk)]
    print(f"  head: {wk} {tuple(sd[wk].shape)} | backbone prefiksi: '{pre}' | e'tiborsiz: {dropped[:6]}")

    m = make_vit(num_classes=N_CLS)
    r = m.load_state_dict(new, strict=False)
    if r.missing_keys or r.unexpected_keys:
        raise RuntimeError(f"[{name}] kalitlar mos kelmadi. missing={r.missing_keys[:5]}, "
                           f"unexpected={r.unexpected_keys[:5]}")
    return m


# %% 5. Modellarni yuklash
print("Modellar yuklanmoqda...")
loaded = {}
for name in ["SimCLR+head", "CE"]:
    print(f"[{name}] {CKPT[name]}")
    loaded[name] = timm_classifier_from_sd(read_state_dict(CKPT[name]), name)

def load_strict(model, sd, name):
    r = model.load_state_dict(sd, strict=False)
    if r.missing_keys or r.unexpected_keys:
        raise RuntimeError(f"[{name}] kalitlar mos kelmadi. missing={r.missing_keys[:5]}, "
                           f"unexpected={r.unexpected_keys[:5]}. Checkpoint kalitlari: {list(sd)[:6]}")
    return model

print("[ArcFace]")
sd = read_state_dict(CKPT["ArcFace"])
n_ck, emb_dim = sd["crit.W"].shape                      # checkpoint ichidagi class soni va embedding o'lchami
print(f"  checkpoint: {n_ck} class, emb_dim={emb_dim}")
loaded["ArcFace"] = load_strict(LitArcFace(n_cls=n_ck, emb_dim=emb_dim, pretrained=False), sd, "ArcFace")

print("[CAC]")
sd = read_state_dict(CKPT["CAC"])
n_ck = sd["proj.weight"].shape[0]
print(f"  checkpoint: {n_ck} class")
loaded["CAC"] = load_strict(LitCAC(n_cls=n_ck, pretrained=False), sd, "CAC")

for m in loaded.values():
    m.to(DEVICE).eval()

model_ncls = {
    "SimCLR+head": loaded["SimCLR+head"].head.out_features,
    "CE": loaded["CE"].head.out_features,
    "ArcFace": loaded["ArcFace"].crit.n_cls,
    "CAC": loaded["CAC"].crit.n_cls,
}
print("\nClass soni:", {"dataset": N_CLS, **model_ncls})
bad = {k: v for k, v in model_ncls.items() if v != N_CLS}
if bad:
    raise RuntimeError(
        f"Class soni mos emas: {bad} (dataset: {N_CLS}). Bu modellar boshqa label mapping bilan o'qitilgan; "
        "indekslar siljigan bo'lishi mumkin, shuning uchun ularni 172 classli mapping bilan qayta o'qiting.")


# Har bir model uchun fn: rasm -> (feature/embedding, native bashorat)
def fn_timm_clf(m):
    def f(x):
        feat = m.forward_head(m.forward_features(x), pre_logits=True)
        return feat, m.head(feat).argmax(1)
    return f


def fn_arcface(m):
    def f(x):
        e = m(x)
        return e, m.crit.cosine(e).argmax(1)             # eng katta cosine
    return f


def fn_cac(m):
    def f(x):
        e = m(x)
        return e, m.crit.dist(e).argmin(1)               # eng yaqin anchor
    return f


MODELS = {
    "SimCLR+head": fn_timm_clf(loaded["SimCLR+head"]),
    "CE": fn_timm_clf(loaded["CE"]),
    "ArcFace": fn_arcface(loaded["ArcFace"]),
    "CAC": fn_cac(loaded["CAC"]),
}


# %% 6. Baholash funksiyalari
@torch.no_grad()
def run(fn, dl):
    feats, preds, ys = [], [], []
    for x, y in dl:
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            f, p = fn(x.to(DEVICE, non_blocking=True))
        feats.append(f.float().cpu())
        preds.append(p.cpu())
        ys.append(y)
    return torch.cat(feats), torch.cat(preds).numpy(), torch.cat(ys).numpy()


def knn_predict(gal_e, gal_y, q_e, k=KNN_K, temp=KNN_T):
    gal = F.normalize(gal_e, dim=1).to(DEVICE)
    gy = torch.as_tensor(gal_y, device=DEVICE)
    out = []
    for i in range(0, len(q_e), 512):
        q = F.normalize(q_e[i:i + 512], dim=1).to(DEVICE)
        w, idx = (q @ gal.T).topk(k, dim=1)
        votes = torch.zeros(len(q), N_CLS, device=DEVICE)
        votes.scatter_add_(1, gy[idx], (w / temp).exp())
        out.append(votes.argmax(1).cpu())
    return torch.cat(out).numpy()


def metrics(t, p):
    # valda mavjud classlar bo'yicha o'rtacha (valda yo'q class o'rtachani sun'iy tushirmasin)
    labels = np.unique(t)
    pm, rm, fm, _ = precision_recall_fscore_support(t, p, labels=labels, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(t, p, labels=labels, average="weighted", zero_division=0)
    return dict(acc=accuracy_score(t, p), prec_macro=pm, rec_macro=rm, f1_macro=fm,
                prec_w=pw, rec_w=rw, f1_w=fw)


def bootstrap_f1_ci(t, p, n=N_BOOT):
    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(t), len(t))
        vals.append(f1_score(t[idx], p[idx], labels=np.unique(t[idx]), average="macro", zero_division=0))
    return np.percentile(vals, [2.5, 97.5])


# %% 7. Hamma modellarni val'da baholash
rows, preds_store, feats_store = [], {}, {}
t_ref = None

for name, fn in MODELS.items():
    print(f"[{name}] baholanmoqda...")
    q_e, p_native, t = run(fn, val_dl)
    if t_ref is None:
        t_ref = t
    assert (t == t_ref).all()                       # hamma modelda val tartibi bir xil
    feats_store[name] = (q_e, t)

    rows.append({"model": name, "protocol": "native", **metrics(t, p_native)})
    preds_store[(name, "native")] = p_native

    g_e, _, g_y = run(fn, gal_dl)                   # kNN: embedding sifati uchun qo'shimcha tahlil
    p_knn = knn_predict(g_e, g_y, q_e)
    rows.append({"model": name, "protocol": "kNN", **metrics(t, p_knn)})
    preds_store[(name, "kNN")] = p_knn

df = pd.DataFrame(rows)
lo_hi = {n: bootstrap_f1_ci(t_ref, preds_store[(n, "native")]) for n in MODELS}
df["f1_macro_ci95"] = df.apply(
    lambda r: f"[{lo_hi[r.model][0]:.4f}, {lo_hi[r.model][1]:.4f}]" if r.protocol == "native" else "", axis=1)
df = df.round(4)
df.to_csv(OUT / "final_comparison.csv", index=False)
print(df.to_string(index=False))


# %% 8. Class bo'yicha F1 va eng ko'p adashishlar
t = t_ref
present = np.unique(t)
per_class = pd.DataFrame(
    {f"{m}-{pr}": f1_score(t, p, labels=present, average=None, zero_division=0)
     for (m, pr), p in preds_store.items()},
    index=[CLASSES[i] for i in present])
per_class.to_csv(OUT / "per_class_f1.csv")
print("\nO'rtacha class F1 (native/kNN):")
print(per_class.mean().sort_values(ascending=False).round(4))

native_df = df[df.protocol == "native"]
best_name = native_df.sort_values("f1_macro", ascending=False).iloc[0]["model"]
cm = confusion_matrix(t, preds_store[(best_name, "native")], labels=range(N_CLS))
np.fill_diagonal(cm, 0)
top = np.dstack(np.unravel_index(np.argsort(cm.ravel())[::-1][:20], cm.shape))[0]
conf = pd.DataFrame([{"true": CLASSES[a], "predicted": CLASSES[b], "count": int(cm[a, b])}
                     for a, b in top if cm[a, b] > 0])
conf.to_csv(OUT / f"top_confusions_{best_name}.csv", index=False)
print(f"\n{best_name}: eng ko'p adashgan juftliklar")
print(conf.head(10).to_string(index=False))


# %% 9. Grafiklar
# 9.1 metrikalar bar chart (native)
fig, ax = plt.subplots(figsize=(9, 4.5))
cols = ["acc", "prec_macro", "rec_macro", "f1_macro"]
x = np.arange(len(cols))
w = 0.8 / len(native_df)
for i, (_, r) in enumerate(native_df.iterrows()):
    ax.bar(x + i * w, [r[c] for c in cols], w, label=r["model"])
ax.set_xticks(x + w * (len(native_df) - 1) / 2)
ax.set_xticklabels(["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"])
ax.set_ylim(0, 1)
ax.set_title("Val dataset: native bashorat")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "metrics_bar.png", dpi=200)
plt.close()

# 9.2 o'qitish egri chiziqlari (logs/<nom>/version_*/metrics.csv)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for f in sorted(glob.glob("logs/*/version_*/metrics.csv")):
    name = Path(f).parents[1].name
    d = pd.read_csv(f)
    for a, col in zip(ax, ["val_f1_macro", "val_acc"]):
        if col in d:
            s = d.dropna(subset=[col]).groupby("epoch")[col].last()
            a.plot(s.index, s.values, label=name)
for a, title in zip(ax, ["Val F1 (macro)", "Val accuracy"]):
    a.set_title(title)
    a.set_xlabel("epoch")
    a.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUT / "training_curves.png", dpi=200)
plt.close()

# 9.3 t-SNE (har bir model embeddingi)
from sklearn.manifold import TSNE

rng = np.random.default_rng(SEED)
sel = rng.choice(len(t), size=min(3000, len(t)), replace=False)
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, (name, (e, y)) in zip(ax, feats_store.items()):
    z = TSNE(n_components=2, init="pca", perplexity=30, random_state=SEED).fit_transform(
        F.normalize(e[sel], dim=1).numpy())
    a.scatter(z[:, 0], z[:, 1], c=y[sel], s=4, cmap="tab20")
    a.set_title(name)
    a.axis("off")
plt.tight_layout()
plt.savefig(OUT / "tsne.png", dpi=200)
plt.close()


# %% 10. Hisobot uchun avtomatik markdown xulosa
def md_table(d):
    head = "| " + " | ".join(d.columns) + " |\n|" + "---|" * len(d.columns) + "\n"
    body = "".join("| " + " | ".join(str(v) for v in r) + " |\n" for r in d.values)
    return head + body


with open(OUT / "summary.md", "w", encoding="utf-8") as f:
    f.write("# Natijalar\n\n")
    f.write(f"Val: {len(val_ds)} rasm, {len(np.unique(t))} class (train: {N_CLS} class)\n\n")
    f.write("## Native bashorat\n\n" + md_table(native_df.drop(columns="protocol")) + "\n")
    f.write("## kNN (embedding sifati)\n\n" + md_table(df[df.protocol == "kNN"].drop(
        columns=["protocol", "f1_macro_ci95"])) + "\n")
    f.write(f"Eng yaxshi model (native macro-F1): **{best_name}**\n")

print(f"\nTayyor. Hamma natijalar '{OUT.resolve()}' papkasida.")

Seed set to 42


train: 35587 rasm | val: 8893 rasm | classlar: 172
Modellar yuklanmoqda...
[SimCLR+head] C:\Users\Administrator\Desktop\Coding\SimCLR_VIT_clf.pth
  [stub] __main__.ViTClf topilmadi, bo'sh klass bilan almashtirildi
  pickle qilingan modul: ViTClf
  head: clf.head.weight (172, 192) | backbone prefiksi: 'clf.' | e'tiborsiz: []
[CE] C:\Users\Administrator\Desktop\Coding\CE_loss_VIT_clf.pth
  pickle qilingan modul: VisionTransformer
  head: head.weight (172, 192) | backbone prefiksi: '' | e'tiborsiz: []
[ArcFace]
  pickle qilingan modul: LitArcFace
  checkpoint: 172 class, emb_dim=128
[CAC]
  pickle qilingan modul: LitCAC
  checkpoint: 172 class

Class soni: {'dataset': 172, 'SimCLR+head': 172, 'CE': 172, 'ArcFace': 172, 'CAC': 172}
[SimCLR+head] baholanmoqda...
[CE] baholanmoqda...
[ArcFace] baholanmoqda...
[CAC] baholanmoqda...
      model protocol    acc  prec_macro  rec_macro  f1_macro  prec_w  rec_w   f1_w    f1_macro_ci95
SimCLR+head   native 0.1496      0.1857     0.1287    0.1009  0